In [ ]:
import os
import glob
import random
import cv2
import matplotlib.pyplot as plt
import sys
import shutil
import yaml
import torch

from collections import defaultdict

try:
    from google.colab import userdata
    IN_COLAB = True
except ImportError:
    userdata = None
    IN_COLAB = False

try:
    from roboflow import Roboflow
except ImportError:
    print("Installing Roboflow library...")
    !pip install roboflow
    from roboflow import Roboflow

try:
    from ultralytics import YOLO
except ImportError:
    print("Installing Ultralytics library...")
    !pip install ultralytics
    from ultralytics import YOLO

try:
    import wandb
except ImportError:
    print("Installing wandb...")
    !pip install wandb
    import wandb

In [ ]:
# Intentar obtener la clave de los Secrets de Colab
def get_api_key():
    if userdata is not None:
        try:
            return userdata.get('RoboflowAPIKey')
        except Exception:
            pass
    return os.environ.get('ROBOFLOW_API_KEY')

try:
    api_key = get_api_key()
    if not api_key:
        raise ValueError("No se encontró la API key. En Colab: configurá el Secret 'RoboflowAPIKey'. Localmente: definí la variable de entorno ROBOFLOW_API_KEY.")

    rf = Roboflow(api_key=api_key)
    project = rf.workspace("raiyan8018").project("sunflower-mn2cr")
    version = project.version(1)
    dataset = version.download("yolov8")

    print(f"Dataset cargado exitosamente en: {dataset.location}")
except Exception as e:
    print(f"Ocurrió un error: {e}")

In [ ]:
# Autenticación con wandb
# Prioridad: 1) Colab Secrets, 2) variable de entorno WANDB_API_KEY, 3) login interactivo
def get_wandb_key():
    if userdata is not None:
        try:
            return userdata.get('WandbAPIKey')
        except Exception:
            pass
    return os.environ.get('WANDB_API_KEY')

wandb_key = get_wandb_key()
if wandb_key != None:
    wandb.login(key=wandb_key)
else:
    wandb.login(relogin=True, force=True)  # prompt interactivo

# Preprocesamiento del dataset

In [ ]:
# Configurar el directorio raíz del proyecto
REPO_NAME = "tp_final_vpc_II_23co"
BRANCH_NAME = "eda"

if IN_COLAB:
    if not os.path.exists(REPO_NAME):
        print(f"Clonando repositorio remoto (rama: {BRANCH_NAME})...")
        !git clone -b {BRANCH_NAME} https://github.com/PabloSalvagni/{REPO_NAME}.git
    else:
        print("Sincronizando últimos cambios de Git...")
        %cd {REPO_NAME}
        !git pull origin {BRANCH_NAME}
        %cd ..

    repo_path = os.path.abspath(REPO_NAME)
    if repo_path not in sys.path:
        sys.path.append(repo_path)
    %cd {REPO_NAME}
else:
    repo_path = os.path.abspath(".")
    if repo_path not in sys.path:
        sys.path.append(repo_path)

print(f"Directorio de trabajo actual: {os.getcwd()}")

### Clonación del dataset

In [ ]:
# Creamos una copia del dataset original para no modificar los datos descargados desde Roboflow
if 'original_dataset_path' not in dir():
    # Fallback: buscar el dataset descargado por Roboflow en el directorio actual
    original_dataset_path = os.path.join(os.getcwd(), "Sunflower-1")
    if not os.path.exists(original_dataset_path):
        raise FileNotFoundError(f"No se encontró el dataset en {original_dataset_path}. Corré primero la celda de descarga.")

if IN_COLAB:
    unified_dataset_path = "/content/sunflower_unified"
else:
    unified_dataset_path = os.path.join(os.path.dirname(original_dataset_path), "sunflower_unified")

if os.path.exists(unified_dataset_path):
    shutil.rmtree(unified_dataset_path)

shutil.copytree(original_dataset_path, unified_dataset_path)

print("Dataset clonado correctamente en:")
print(unified_dataset_path)

### Unificación de clases

In [ ]:
# Unificamos las clases del dataset: cualquier class_id pasa a ser 0
for split in ["train", "valid", "test"]:
    labels_dir = os.path.join(unified_dataset_path, split, "labels")

    for label_file in os.listdir(labels_dir):
        label_path = os.path.join(labels_dir, label_file)

        new_lines = []

        with open(label_path, "r") as file:
            lines = file.readlines()

        for line in lines:
            parts = line.strip().split()

            if len(parts) == 5:
                parts[0] = "0"
                new_lines.append(" ".join(parts))

        with open(label_path, "w") as file:
            file.write("\n".join(new_lines))

print("Clases unificadas correctamente.")

In [ ]:
# Actualizamos el archivo data.yaml para indicar que ahora existe una sola clase
yaml_path = os.path.join(unified_dataset_path, "data.yaml")

with open(yaml_path, "r") as file:
    data_yaml = yaml.safe_load(file)

data_yaml["nc"] = 1
data_yaml["names"] = ["sunflower"]

with open(yaml_path, "w") as file:
    yaml.dump(data_yaml, file, sort_keys=False)

print("data.yaml actualizado correctamente.")

In [ ]:
# Verificamos que todos los labels tengan únicamente class_id = 0
class_ids = set()

for split in ["train", "valid", "test"]:
    labels_dir = os.path.join(unified_dataset_path, split, "labels")

    for label_file in os.listdir(labels_dir):
        label_path = os.path.join(labels_dir, label_file)

        with open(label_path, "r") as file:
            for line in file:
                parts = line.strip().split()

                if len(parts) == 5:
                    class_id = int(parts[0])
                    class_ids.add(class_id)

print("Clases presentes después de unificar:")
print(class_ids)

In [ ]:
# Verificamos nuevamente cantidad de imágenes y labels en cada partición del dataset unificado
for split in ["train", "valid", "test"]:
    images_dir = os.path.join(unified_dataset_path, split, "images")
    labels_dir = os.path.join(unified_dataset_path, split, "labels")

    n_images = len(os.listdir(images_dir))
    n_labels = len(os.listdir(labels_dir))

    print(f"{split}: {n_images} imágenes | {n_labels} labels")

# Entrenamiento

In [ ]:
def with_wandb_logging(project, name):
    def decorator(train_fn):
        def wrapper(*args, **kwargs):
            run = wandb.init(project=project, name=name)

            model = kwargs.get("model") or args[0]

            def log_epoch(trainer):
                wandb.log({**trainer.metrics, "epoch": trainer.epoch})

            model.add_callback("on_train_epoch_end", log_epoch)

            metrics = train_fn(*args, **kwargs)

            wandb.log(metrics.results_dict)
            wandb.save(f"{project}/{name}/weights/best.pt")
            wandb.finish()

            return metrics
        return wrapper
    return decorator

In [ ]:
torch.cuda.empty_cache()
device_to_use = 0 if torch.cuda.is_available() else "cpu"

model = YOLO("yolov8n.pt")

@with_wandb_logging(project="vpc2", name="yolov8n_baseline")
def train(model, **kwargs):
    return model.train(**kwargs)

metrics = train(
    model=model,
    data=os.path.join(unified_dataset_path, "data.yaml"),
    epochs=20,
    imgsz=640,
    batch=16,
    device=device_to_use,
    workers=2,
    project="vpc2",
    name="yolov8n_baseline",
    save=True,
    plots=True,
)

del model
torch.cuda.empty_cache()

In [ ]:
torch.cuda.empty_cache()
device_to_use = 0 if torch.cuda.is_available() else "cpu"

model = YOLO("yolov8s.pt")

@with_wandb_logging(project="vpc2", name="yolov8s_baseline")
def train(model, **kwargs):
    return model.train(**kwargs)

metrics = train(
    model=model,
    data=os.path.join(unified_dataset_path, "data.yaml"),
    epochs=20,
    imgsz=640,
    batch=16,
    device=device_to_use,
    workers=2,
    project="vpc2",
    name="yolov8s_baseline",
    save=True,
    plots=True,
)

del model
torch.cuda.empty_cache()

In [ ]:
torch.cuda.empty_cache()
device_to_use = 0 if torch.cuda.is_available() else "cpu"

model = YOLO("yolov8m.pt")

@with_wandb_logging(project="vpc2", name="yolov8m_baseline")
def train(model, **kwargs):
    return model.train(**kwargs)

metrics = train(
    model=model,
    data=os.path.join(unified_dataset_path, "data.yaml"),
    epochs=20,
    imgsz=640,
    batch=16,
    device=device_to_use,
    workers=2,
    project="vpc2",
    name="yolov8m_baseline",
    save=True,
    plots=True,
)

del model
torch.cuda.empty_cache()

In [ ]:
torch.cuda.empty_cache()
device_to_use = 0 if torch.cuda.is_available() else "cpu"

model = YOLO("yolov10n.pt")

@with_wandb_logging(project="vpc2", name="yolov10n_baseline")
def train(model, **kwargs):
    return model.train(**kwargs)

metrics = train(
    model=model,
    data=os.path.join(unified_dataset_path, "data.yaml"),
    epochs=20,
    imgsz=640,
    batch=-1,
    device=device_to_use,
    workers=2,
    project="vpc2",
    name="yolov10n_baseline",
    save=True,
    plots=True,
)

del model
torch.cuda.empty_cache()

In [ ]:
torch.cuda.empty_cache()
device_to_use = 0 if torch.cuda.is_available() else "cpu"

model = YOLO("yolo11n.pt")

@with_wandb_logging(project="vpc2", name="yolo11n_baseline")
def train(model, **kwargs):
    return model.train(**kwargs)

metrics = train(
    model=model,
    data=os.path.join(unified_dataset_path, "data.yaml"),
    epochs=20,
    imgsz=640,
    batch=-1,
    device=device_to_use,
    workers=2,
    project="vpc2",
    name="yolo11n_baseline",
    save=True,
    plots=True,
)

del model
torch.cuda.empty_cache()

In [ ]:
torch.cuda.empty_cache()
device_to_use = 0 if torch.cuda.is_available() else "cpu"

model = YOLO("yolo11s.pt")

@with_wandb_logging(project="vpc2", name="yolo11s_baseline")
def train(model, **kwargs):
    return model.train(**kwargs)

metrics = train(
    model=model,
    data=os.path.join(unified_dataset_path, "data.yaml"),
    epochs=20,
    imgsz=640,
    batch=-1,
    device=device_to_use,
    workers=2,
    project="vpc2",
    name="yolo11s_baseline",
    save=True,
    plots=True,
)

del model
torch.cuda.empty_cache()

In [ ]:
torch.cuda.empty_cache()
device_to_use = 0 if torch.cuda.is_available() else "cpu"

model = YOLO("yolo26s.pt")

@with_wandb_logging(project="vpc2", name="yolo26s_baseline")
def train(model, **kwargs):
    return model.train(**kwargs)

metrics = train(
    model=model,
    data=os.path.join(unified_dataset_path, "data.yaml"),
    epochs=20,
    imgsz=640,
    batch=-1,
    device=device_to_use,
    workers=2,
    project="vpc2",
    name="yolo26s_baseline",
    save=True,
    plots=True,
)

del model
torch.cuda.empty_cache()

In [ ]:
torch.cuda.empty_cache()
device_to_use = 0 if torch.cuda.is_available() else "cpu"

model = YOLO("yolo11s.pt")

@with_wandb_logging(project="vpc2", name="yolo11s_sgd")
def train(model, **kwargs):
    return model.train(**kwargs)

metrics = train(
    model=model,
    data=os.path.join(unified_dataset_path, "data.yaml"),
    epochs=20,
    imgsz=640,
    batch=-1,
    device=device_to_use,
    workers=2,
    project="vpc2",
    name="yolo11s_sgd",
    optimizer="SGD",
    lr0=0.01,
    momentum=0.937,
    weight_decay=0.0005,
    save=True,
    plots=True,
)

del model
torch.cuda.empty_cache()